# Simulations with a time limit (Section 4.6)

This notebook reproduces Figures 6-8 (Section 4), where each SMA run stops either at the theoretical J target or after a fixed wall-clock limit, whichever happens first. We consider two time budgets: 5 minutes and 15 minutes.


In [ ]:
import sys, pickle
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scripts.sma import delta_pi_numba
from scripts.parameters import epsilon_target
from scripts.experiment import run_budgeted_experiment
from scripts.diagnostics import certify_worst_case_risk

root_folder = Path.cwd().parent
data_path = root_folder / 'data'
five_min_results = data_path / 'budgeted_results_300s.pkl'
fifteen_min_results = data_path / 'budgeted_results_900s.pkl'

## Run (or load) the 5-minute and 15-minute simulations

In [ ]:
rerun_budgeted_simulations = False

def run_with_time_limit(time_budget, m_values, cache_path):
    results = []
    for m in m_values:
        eps = epsilon_target(m)
        print(f"\nm = {m:.1f}, eps = {eps:.4f}, budget = {time_budget:.0f}s")
        res = run_budgeted_experiment(m, eps, time_budget=time_budget)
        results.append(res)
        status = "DONE" if res['completed'] else f"STOPPED at {res['pct_completed']:.1f}%"
        print(f"J: {res['J_completed']:,} / {res['J_target']:,} ({status})")
        print(f"lower = {res['bayes_risk']:.4f}, sup_risk = {res['worst_case_risk']:.4f}")
        print(f"runtime = {res['smd_runtime']:.1f}s")
    with open(cache_path, 'wb') as f:
        pickle.dump(results, f)
    return results

m_values = np.arange(1.0, 4.01, 0.2)

In [ ]:
# 5-minute limit
if rerun_budgeted_simulations or not five_min_results.exists():
    budgeted_results = run_with_time_limit(300, m_values, five_min_results)
else:
    with open(five_min_results, 'rb') as f:
        budgeted_results = pickle.load(f)
    print(f"Loaded 5-minute budget results from {five_min_results}")

In [ ]:
# 15-minute limit
if rerun_budgeted_simulations or not fifteen_min_results.exists():
    budgeted_results_15 = run_with_time_limit(900, m_values, fifteen_min_results)
else:
    with open(fifteen_min_results, 'rb') as f:
        budgeted_results_15 = pickle.load(f)
    print(f"Loaded 15-minute budget results from {fifteen_min_results}")

## Figure 6 - Required vs completed iterations (5-minute limit)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

m_vals = [r['m'] for r in budgeted_results]
j_done = [r['J_completed'] for r in budgeted_results]
j_target = [r['J_target'] for r in budgeted_results]

ax.plot(m_vals, np.array(j_target) / 1e6, 's--', color='dimgray', markersize=5, linewidth=1.5, label=r'$J_{\mathrm{required}}(m)$')
ax.plot(m_vals, np.array(j_done) / 1e6, 'o-', color='crimson', markersize=5, linewidth=1.5, label=r'$J_{\mathrm{completed}}(m)$')
ax.fill_between(m_vals, np.array(j_done) / 1e6, np.array(j_target) / 1e6, where=np.array(j_done) < np.array(j_target), interpolate=True, alpha=0.15, color='red', label='Time constraint binds')
ax.set_xlabel(r'$m$', fontsize=13)
ax.set_ylabel('J(m) (millions)', fontsize=13)
ax.legend(fontsize=10)
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.show()

## Figure 7 - Risk under the 5-minute limit vs linear minimax

In [ ]:
n_risk_mc   = 1000
n_eval_cert = 200000 # here we use a fine grid for the certified bound again
noise = np.random.default_rng(123).standard_normal(n_risk_mc)

certified = {}
for name, results in [('5min', budgeted_results), ('15min', budgeted_results_15)]:
    m_plot, lower_bound_plot, v_linear_plot, sup_risk_sma_plot = [], [], [], []
    for res in results:                      
        m          = res['m']
        theta_grid = res['theta_grid']
        avg_prior  = res['avg_prior']
        theta_eval = np.linspace(-m, m, n_eval_cert)
        Y_eval     = noise[None, :] + theta_eval[:, None]
        d_eval     = delta_pi_numba(Y_eval.ravel(), theta_grid, avg_prior).reshape(Y_eval.shape)
        risk_eval  = np.mean((d_eval - theta_eval[:, None])**2, axis=1)
        cert       = certify_worst_case_risk(risk_eval, theta_eval, m)
        m_plot.append(m)
        v_linear_plot.append(m**2 / (1 + m**2))
        lower_bound_plot.append(0.8 * m**2 / (1 + m**2))
        sup_risk_sma_plot.append(cert['certified_upper'])
    certified[name] = tuple(map(np.array, (m_plot, lower_bound_plot, v_linear_plot, sup_risk_sma_plot)))

(m_plot,    lower_bound_plot,    v_linear_plot,    sup_risk_sma_plot)    = certified['5min']
(m_plot_15, lower_bound_plot_15, v_linear_plot_15, sup_risk_sma_plot_15) = certified['15min']

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(m_plot, lower_bound_plot, '-', color='orange', markersize=5, linewidth=2, label=r'$\frac{1}{1.25}\,\bar{v}_{\mathrm{linear}}(m)$')
ax.plot(m_plot, v_linear_plot, '-', color='dimgray', markersize=5, linewidth=1.5, label=r'$\bar{v}_{\mathrm{linear}}(m)$')
ax.plot(m_plot, sup_risk_sma_plot, 'o-', color='crimson', markersize=5, linewidth=1.5, label=r'$\sup_\theta R(d_{\pi^\epsilon}, \theta)$ (certified)')
ax.axvline(2.2, color='gray', linestyle=':', alpha=0.5, label='5 min limit binds')
ax.set_xlabel(r'$m$', fontsize=13)
ax.set_ylabel('Risk', fontsize=13)
ax.legend(fontsize=9, loc='lower right')
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.show()

## Figure 8 - Risk improvement over linear (5 min and 15 min)

In [ ]:
mask_5 = v_linear_plot > 1e-12
pct_sup = np.full_like(v_linear_plot, np.nan, dtype=float)
pct_sup[mask_5] = 100.0 * (1.0 - sup_risk_sma_plot[mask_5] / v_linear_plot[mask_5])
mask_15 = v_linear_plot_15 > 1e-12
pct_sup_15 = np.full_like(v_linear_plot_15, np.nan, dtype=float)
pct_sup_15[mask_15] = 100.0 * (1.0 - sup_risk_sma_plot_15[mask_15] / v_linear_plot_15[mask_15])

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(m_plot, pct_sup, 'o-', color='crimson', linewidth=1.8, markersize=5, label=r'$\sup_\theta R(d_{\pi^\epsilon},\theta)$ (certified), 5 min')
ax.plot(m_plot_15, pct_sup_15, 's--', color='crimson', linewidth=1.8, markersize=5, label=r'$\sup_\theta R(d_{\pi^\epsilon},\theta)$ (certified), 15 min')
ax.axhline(20, linestyle='--', color='orange', alpha=1, label=r'$20\%$ improvement')
ax.axhline(0, linestyle=':', color='black', alpha=0.7)
ax.axvline(2.2, color='gray', linestyle=':', alpha=0.5, label='5 min limit binds')
ax.axvline(2.4, color='gray', linestyle='-.', alpha=0.5,label='15 min limit binds')
ax.set_xlabel(r'$m$', fontsize=13)
ax.set_ylabel('Improvement over linear estimator (%)', fontsize=13)
ax.legend(fontsize=9, loc='best')
plt.tight_layout()
plt.show()